# Sending results to your tracking tools

The same results, logged to MLflow, Weights & Biases (offline) and OpenTelemetry, all to local folders or memory, so nothing leaves your machine. It needs `pip install "llmdyno[mlflow,wandb,otel,data]"`. With a tracking server, pass its URL instead.

In [ ]:
import tempfile

import llmdyno

# A throwaway data folder, so this notebook never touches yours. Use llmdyno.init() (no argument) for your own.
llmdyno.init(tempfile.mkdtemp(prefix="llmdyno-notebook-"))
plan_id = llmdyno.load_sample()          # 15 configurations measured on an RTX 5080, bundled with LLMDyno
results = llmdyno.results(plan_id)
results

## MLflow: one run per configuration

In [ ]:
import os
import pathlib

folder = pathlib.Path(tempfile.mkdtemp(prefix="llmdyno-tracking-"))
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"     # MLflow 3 keeps the folder store for local use
ids = results.log_to_mlflow(experiment="llmdyno-demo", tracking_uri=(folder / "mlruns").as_uri())
len(ids)

In [ ]:
import mlflow

mlflow.set_tracking_uri((folder / "mlruns").as_uri())
runs = mlflow.search_runs(experiment_names=["llmdyno-demo"])
runs[["tags.mlflow.runName", "metrics.final", "metrics.correctness", "params.harness"]].sort_values("metrics.final", ascending=False).head()

## Weights & Biases, offline

`wandb sync` uploads the run later if you want it in your W&B project.

In [ ]:
os.environ["WANDB_SILENT"] = "true"
results.log_to_wandb(project="llmdyno-demo", mode="offline", dir=str(folder))

## OpenTelemetry

Each run becomes a trace (GenAI semantic conventions). Here the spans go to memory; point `to_otel(endpoint=...)` at Phoenix, Langfuse, Datadog or any OTLP collector instead.

In [ ]:
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import SimpleSpanProcessor
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter

memory = InMemorySpanExporter()
provider = TracerProvider()
provider.add_span_processor(SimpleSpanProcessor(memory))
results.to_otel(tracer_provider=provider)
span = memory.get_finished_spans()[0]
span.name, dict(span.attributes)